# Evaluation and Inference

**Objective**: Load a trained Graph-Transformer checkpoint and evaluate on the test set using greedy decoding and beam search. Display sample predictions alongside ground truth.

This notebook serves as the final stage of the case study.

In [ ]:
import math, os, sys, re
from pathlib import Path

import sympy
from sympy.parsing.sympy_parser import parse_expr
import threading

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import to_dense_batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Batch
import torch_geometric.transforms as T
from tqdm.notebook import tqdm
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent if 'Notebooks' in os.getcwd() else Path(os.getcwd())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Project root: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')

## 1 - Configuration and Model Definition

In [ ]:
D_MODEL       = 256
GNN_HIDDEN    = 256
GNN_LAYERS    = 3
N_HEADS       = 8
N_DEC_LAYERS  = 4
D_FF          = 1024
DROPOUT       = 0.1
MAX_LEN       = 512
NODE_FEAT_DIM = 64
EDGE_FEAT_DIM = 32
RWSE_DIM      = 16
MAX_NODES     = 200
BATCH         = 32
PAD_IDX       = 1

class GraphTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.node_emb = nn.Linear(NODE_FEAT_DIM, GNN_HIDDEN)
        self.idx_emb  = nn.Embedding(MAX_NODES, GNN_HIDDEN)
        self.rwse_emb = nn.Linear(RWSE_DIM, GNN_HIDDEN)
        self.convs = nn.ModuleList([
            TransformerConv(GNN_HIDDEN, GNN_HIDDEN, heads=4, concat=False, edge_dim=EDGE_FEAT_DIM, dropout=DROPOUT)
            for _ in range(GNN_LAYERS)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(GNN_HIDDEN) for _ in range(GNN_LAYERS)])
        self.proj  = nn.Linear(GNN_HIDDEN, D_MODEL)

    def forward(self, batch_data):
        x, edge_index, batch, edge_attr = batch_data.x, batch_data.edge_index, batch_data.batch, batch_data.edge_attr
        h = self.node_emb(x)
        counts = torch.bincount(batch)
        local_idx = torch.cat([torch.arange(c.item(), device=x.device) for c in counts])
        h = h + self.idx_emb(local_idx)
        if hasattr(batch_data, 'rwse'):
            h = h + self.rwse_emb(batch_data.rwse)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h = norm(h + h_new)
        h_dense, mask = to_dense_batch(h, batch)
        return self.proj(h_dense), mask

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(-torch.arange(0, d_model, 2).float() * math.log(10000.0) / d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AdvancedGraphTransformer(nn.Module):
    def __init__(self, tgt_vocab_size):
        super().__init__()
        self.encoder  = GraphTransformerEncoder()
        self.tgt_emb  = nn.Embedding(tgt_vocab_size, D_MODEL, padding_idx=PAD_IDX)
        self.pos_enc  = PositionalEncoding(D_MODEL)
        dec_layer = nn.TransformerDecoderLayer(D_MODEL, N_HEADS, D_FF, DROPOUT, batch_first=True, norm_first=True)
        self.decoder  = nn.TransformerDecoder(dec_layer, N_DEC_LAYERS)
        self.out_proj = nn.Linear(D_MODEL, tgt_vocab_size)

    def forward(self, batch_data, tgt, tgt_mask=None, tgt_pad_mask=None):
        memory, mem_mask = self.encoder(batch_data)
        t = self.pos_enc(self.tgt_emb(tgt))
        out = self.decoder(t, memory, tgt_mask=tgt_mask,
                           tgt_key_padding_mask=tgt_pad_mask,
                           memory_key_padding_mask=torch.logical_not(mem_mask))
        return self.out_proj(out)

print('Model architecture defined.')

## 2 - Load Data and Checkpoint

In [ ]:
data_dir = PROJECT_ROOT / 'preprocessed' / 'qed'
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'advanced_graph_mlp_qed_best.pt'

test_data = torch.load(data_dir / 'test_graphs.pt')
test_ds, vocab = test_data['dataset'], test_data['vocab']

rwse = T.AddRandomWalkPE(walk_length=RWSE_DIM, attr_name='rwse')
for i in range(len(test_ds)):
    test_ds[i] = rwse(test_ds[i])

test_loader = DataLoader(test_ds, batch_size=BATCH)

model = AdvancedGraphTransformer(len(vocab)).to(DEVICE)

if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f'Loaded Checkpoint: {ckpt_path}')
else:
    print(f'Warning: Checkpoint not found at {ckpt_path}. Model specifies random weights.')

model.eval()

bos_idx = vocab.token_to_idx.get('<bos>', 2)
eos_idx = vocab.token_to_idx.get('<eos>', 3)

print(f'Test samples: {len(test_ds)}')
print(f'Vocab size  : {len(vocab)}')

## 3 - Decoding Functions

In [ ]:
def causal_mask(sz, device):
    return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()

@torch.no_grad()
def greedy_decode_batch(model, batch_data, bos_idx, eos_idx, pad_idx, max_len, device):
    model.eval()
    memory, mem_mask = model.encoder(batch_data)
    mem_key_pad = torch.logical_not(mem_mask)
    batch_size = memory.size(0)
    ys = torch.full((batch_size, 1), bos_idx, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
    for _ in range(max_len - 1):
        tgt_mask = causal_mask(ys.size(1), device)
        t = model.pos_enc(model.tgt_emb(ys))
        out = model.decoder(t, memory, tgt_mask=tgt_mask,
                            memory_key_padding_mask=mem_key_pad)
        logits = model.out_proj(out[:, -1, :])
        next_tok = logits.argmax(-1)
        next_tok = next_tok.masked_fill(finished, pad_idx)
        ys = torch.cat([ys, next_tok.unsqueeze(1)], dim=1)
        finished = finished | (next_tok == eos_idx)
        if finished.all():
            break
    return ys[:, 1:]

@torch.no_grad()
def beam_search_single(model, batch_data, bos_idx, eos_idx, max_len, k, device):
    model.eval()
    memory, mem_mask = model.encoder(batch_data)
    mem_key_pad = torch.logical_not(mem_mask)
    beams = [(0.0, [bos_idx])]
    for _ in range(max_len):
        candidates = []
        for score, seq in beams:
            if seq[-1] == eos_idx:
                candidates.append((score, seq))
                continue
            ys = torch.tensor([seq], dtype=torch.long, device=device)
            tgt_mask = causal_mask(ys.size(1), device)
            t = model.pos_enc(model.tgt_emb(ys))
            out = model.decoder(t, memory, tgt_mask=tgt_mask,
                                memory_key_padding_mask=mem_key_pad)
            log_probs = torch.log_softmax(model.out_proj(out[0, -1]), dim=-1)
            top_probs, top_ids = log_probs.topk(k)
            for prob, idx in zip(top_probs.tolist(), top_ids.tolist()):
                candidates.append((score + prob, seq + [idx]))
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:k]
        if all(s[-1] == eos_idx for _, s in beams):
            break
    return beams[0][1][1:]

def check_symbolic_equivalence(p_toks, t_toks):
    def toks_to_str(l):
        parts = []
        for t in l:
            tok = str(t)
            if tok in ('<bos>', '<eos>', '<pad>', '<unk>'):
                continue
            tok = tok.replace('<S>', 's_var').replace('<s>', 's_var')
            tok = tok.replace('<T>', 't_var').replace('<t>', 't_var')
            tok = tok.replace('<U>', 'u_var').replace('<u>', 'u_var')
            tok = tok.replace('<GAMMA>', 'gamma_var').replace('<gamma>', 'gamma_var')
            tok = tok.replace('SQUARE', '**2')
            tok = tok.replace('INDEX_', 'idx_')
            tok = tok.replace('MOMENTUM_', 'p_')
            tok = tok.replace('<', '').replace('>', '')
            parts.append(tok)
        return ' '.join(parts)
    p, t = toks_to_str(p_toks), toks_to_str(t_toks)
    if not p.strip() or not t.strip(): return False
    p_norm = re.sub(r'\s+', '', p)
    t_norm = re.sub(r'\s+', '', t)
    if p_norm == t_norm: return True
    result = [False]
    def _check():
        try:
            local_dict = {}
            expr_p = parse_expr(p, local_dict=local_dict)
            expr_t = parse_expr(t, local_dict=local_dict)
            if sympy.expand(expr_p - expr_t) == 0:
                result[0] = True
                return
            result[0] = (sympy.simplify(expr_p - expr_t) == 0)
        except Exception:
            pass
    worker = threading.Thread(target=_check)
    worker.daemon = True
    worker.start()
    worker.join(5.0)
    return result[0]

print('Decoding functions defined.')

## 4 - Greedy Evaluation on Test Set

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
total_loss, c_seq, c_sym, n_seq = 0.0, 0, 0, 0

for batch in tqdm(test_loader):
    batch = batch.to(DEVICE)
    tgt_in  = batch.y[:, :-1]
    tgt_out = batch.y[:, 1:]
    with torch.no_grad():
        tgt_mask = causal_mask(tgt_in.size(1), DEVICE)
        tgt_pad  = torch.eq(tgt_in, PAD_IDX)
        logits   = model(batch, tgt_in, tgt_mask=tgt_mask, tgt_pad_mask=tgt_pad)
        loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        total_loss += loss.item()
        preds = greedy_decode_batch(model, batch, bos_idx, eos_idx, PAD_IDX, MAX_LEN, DEVICE)
        for b in range(preds.shape[0]):
            p_eos = (preds[b] == eos_idx).nonzero(as_tuple=True)[0]
            p_len = p_eos[0].item() if len(p_eos) > 0 else preds.shape[1]
            t_eos = (tgt_out[b] == eos_idx).nonzero(as_tuple=True)[0]
            t_len = t_eos[0].item() if len(t_eos) > 0 else tgt_out.shape[1]
            p_seq = preds[b, :p_len]
            t_seq = tgt_out[b, :t_len]
            if p_len == t_len and torch.equal(p_seq, t_seq):
                c_seq += 1
            else:
                ptoks = [vocab.idx_to_token.get(i.item(), '') for i in p_seq]
                ttoks = [vocab.idx_to_token.get(i.item(), '') for i in t_seq]
                if check_symbolic_equivalence(ptoks, ttoks):
                    c_sym += 1
            n_seq += 1

greedy_seq = c_seq / n_seq * 100 if n_seq else 0
greedy_sym = (c_seq + c_sym) / n_seq * 100 if n_seq else 0

print('=' * 60)
print('  GREEDY DECODING Results')
print('=' * 60)
print(f'  Test loss : {total_loss / len(test_loader):.4f}')
print(f'  seq_acc   : {greedy_seq:.2f}%')
print(f'  sym_acc   : {greedy_sym:.2f}%')

## 5 - Beam Search Evaluation (k=5)

In [ ]:
BEAM_K = 5
c_seq_b, c_sym_b, n_seq_b = 0, 0, 0

for batch in tqdm(test_loader):
    data_list = batch.to_data_list()
    for sample in data_list:
        sample = sample.to(DEVICE)
        single_batch = Batch.from_data_list([sample])
        tgt_out = sample.y[0, 1:]
        best_seq = beam_search_single(model, single_batch, bos_idx, eos_idx, MAX_LEN, BEAM_K, DEVICE)
        best_seq_tensor = torch.tensor(best_seq, device=DEVICE)
        t_eos = (tgt_out == eos_idx).nonzero(as_tuple=True)[0]
        t_len = t_eos[0].item() if len(t_eos) > 0 else len(tgt_out)
        t_seq = tgt_out[:t_len]
        p_eos = (best_seq_tensor == eos_idx).nonzero(as_tuple=True)[0]
        p_len = p_eos[0].item() if len(p_eos) > 0 else len(best_seq_tensor)
        p_seq = best_seq_tensor[:p_len]
        if p_len == t_len and torch.equal(p_seq, t_seq):
            c_seq_b += 1
        else:
            ptoks = [vocab.idx_to_token.get(i.item(), '') for i in p_seq]
            ttoks = [vocab.idx_to_token.get(i.item(), '') for i in t_seq]
            if check_symbolic_equivalence(ptoks, ttoks):
                c_sym_b += 1
        n_seq_b += 1

beam_seq = c_seq_b / n_seq_b * 100 if n_seq_b else 0
beam_sym = (c_seq_b + c_sym_b) / n_seq_b * 100 if n_seq_b else 0

print('=' * 60)
print(f'  BEAM SEARCH (k={BEAM_K}) Results')
print('=' * 60)
print(f'  seq_acc : {beam_seq:.2f}%')
print(f'  sym_acc : {beam_sym:.2f}%')

## 6 - Sample Predictions

In [ ]:
def decode_ids(ids, vocab, eos_idx):
    tokens = []
    for i in ids:
        idx = i.item() if hasattr(i, 'item') else i
        if idx == eos_idx:
            break
        tok = vocab.idx_to_token.get(idx, '<unk>')
        if tok not in ('<bos>', '<pad>'):
            tokens.append(tok)
    return ' '.join(tokens)

N_SAMPLES = 10

for i in range(min(N_SAMPLES, len(test_ds))):
    sample = test_ds[i].to(DEVICE)
    single_batch = Batch.from_data_list([sample])
    tgt_out = sample.y[0, 1:]
    greedy_preds = greedy_decode_batch(model, single_batch, bos_idx, eos_idx, PAD_IDX, MAX_LEN, DEVICE)
    beam_preds = beam_search_single(model, single_batch, bos_idx, eos_idx, MAX_LEN, 5, DEVICE)
    truth   = decode_ids(tgt_out, vocab, eos_idx)
    greedy  = decode_ids(greedy_preds[0], vocab, eos_idx)
    beam    = decode_ids(beam_preds, vocab, eos_idx)
    match_g = 'MATCH' if greedy == truth else 'MISS'
    match_b = 'MATCH' if beam == truth else 'MISS'
    print(f'--- Sample {i+1} ---')
    print(f'  Truth  : {truth[:100]}...')
    print(f'  Greedy : {greedy[:100]}... [{match_g}]')
    print(f'  Beam(5): {beam[:100]}... [{match_b}]')
    print()

## 7 - Results Summary

In [ ]:
print('=' * 60)
print('  FINAL TEST RESULTS')
print('=' * 60)
print(f'  Greedy  : seq_acc = {greedy_seq:.2f}%   sym_acc = {greedy_sym:.2f}%')
print(f'  Beam(5) : seq_acc = {beam_seq:.2f}%   sym_acc = {beam_sym:.2f}%')
print(f'  Test samples evaluated: {n_seq}')